# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliLabib2006/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from pathlib import Path
import os
import subprocess

In [1]:
!git clone https://github.com/AliLabib2006/flyrank-internship-ml

Cloning into 'flyrank-internship-ml'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 127 (delta 40), reused 92 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.85 MiB | 12.48 MiB/s, done.
Resolving deltas: 100% (40/40), done.


In [2]:
%cd flyrank-internship-ml

/content/flyrank-internship-ml


In [3]:
!ls

02_your_first_readable_model.ipynb  docs       README.md	 submission
AGENTS.md			    GUIDE.md   requirements.txt  work
CLAUDE.md			    LICENSE    scripts
data				    notebooks  SETUP.md
DATA_USE.md			    outputs    skills


In [4]:
!ls data/raw

content_refresh_anonymized.csv


In [5]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(df.shape)
df.head()

(30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


## 1. My lane as an ML task (type)

I selected **Lane 2: Refresh / Content Opportunity Scoring**.

This is mainly a **ranking and scoring task** because the goal is to give each content page a priority score and arrange the pages from highest priority to lowest priority.

The final output will be a ranked review queue that helps a content or SEO reviewer decide which pages to inspect first. A classification model may estimate the probability that a page is declining, but the final business output is still a ranking rather than only a yes-or-no prediction.

After reviewing a recommended page, the reviewer may refresh, expand, protect, merge, prune, or monitor it.

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane = "Refresh / Content Opportunity Scoring"
task_type = "Ranking and scoring"
unit = "One content page"
output = "A ranked page-review queue"

print("Lane:", lane)
print("ML task type:", task_type)
print("Unit of analysis:", unit)
print("Output:", output)


Lane: Refresh / Content Opportunity Scoring
ML task type: Ranking and scoring
Unit of analysis: One content page
Output: A ranked page-review queue


## 2. Target or proxy

My starter target will be a binary proxy called **`is_declining_proxy`**.

It will have two values:

- `1` when the page's `trend_direction` is `down`;
- `0` when the page's `trend_direction` is not `down`.

This is a **defined proxy**, not a true future observed outcome. It describes
the page's current 90-day trend, so it cannot prove that the page will decline
in the future.

A stronger target later would use separate time periods:

**previous 90 days of page signals → decline during the following 30 days**

Because this proxy is created from `trend_direction`, I will not use
`trend_direction` or `trend_pct` as model features. That would leak the answer
into the model.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd

target_example = pd.DataFrame({
    "trend_direction": [
        "down",
        "up",
        "stable",
        "down"
    ]
})

target_example["is_declining_proxy"] = (
    target_example["trend_direction"]
    .str.lower()
    .eq("down")
    .astype(int)
)

target_example


,trend_direction,is_declining_proxy
0,down,1
1,up,0
2,stable,0
3,down,1


## 3. Success metric

My main success metric will be **Precision@50**.

Precision@50 measures how many of the top 50 pages recommended by the system
match the decline proxy.

For example, a Precision@50 of `0.50` means that 25 of the top 50 recommended
pages are marked as declining.

This metric fits the real decision because a content or SEO reviewer has
limited time and may only inspect the highest-ranked pages.

I will consider the method useful only if it performs better than a simple,
transparent rule-based baseline. My provisional goal is a Precision@50 of at
least `0.50`.

This metric measures the quality of the ranking. It does not prove that
editing a recommended page will cause its performance to improve.

In [9]:
k = 50
precision_goal = 0.50

matching_pages = int(k * precision_goal)

print("Success metric: Precision@50")
print("Provisional goal:", precision_goal)
print(
    f"This means at least {matching_pages} "
    f"of the top {k} pages match the proxy."
)

Success metric: Precision@50
Provisional goal: 0.5
This means at least 25 of the top 50 pages match the proxy.


## 4. The unit of analysis, as a real dataframe

The unit of analysis is **one pseudonymized content page**.

Each row in the starter dataset represents one content item. For this lane, I
will keep pages that have more than zero impressions, are at least 90 days
old, and are unique by `content_id`.

The dataframe below shows safe page-level measurements such as impressions,
clicks, sessions, content age, CTR, average position, engagement, and the
decline proxy.

The pseudonymized `content_id` is used only to identify and remove duplicate
rows. It will not be used as a model feature.

In [10]:
from pathlib import Path
import pandas as pd

data_path = Path("data/raw/content_refresh_anonymized.csv")

if not data_path.exists():
    raise FileNotFoundError(
        "The dataset was not found. Make sure the repository is cloned "
        "and that you are inside the repository folder."
    )

df = pd.read_csv(data_path)

lane_df = (
    df.loc[
        (df["impressions_90d"] > 0)
        & (df["content_age_days"] >= 90)
    ]
    .drop_duplicates(subset="content_id")
    .copy()
)

lane_df["is_declining_proxy"] = (
    lane_df["trend_direction"]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("down")
    .astype(int)
)

wanted_columns = [
    "content_id",
    "impressions_90d",
    "clicks_90d",
    "sessions_90d",
    "content_age_days",
    "ctr",
    "avg_position",
    "engagement_rate",
    "trend_direction",
    "is_declining_proxy",
]

display_columns = [
    column
    for column in wanted_columns
    if column in lane_df.columns
]

print("Unit of analysis: one pseudonymized content page")
print("Number of page-level rows:", f"{len(lane_df):,}")

lane_df[display_columns].head(10)

Unit of analysis: one pseudonymized content page
Number of page-level rows: 30,000


,content_id,impressions_90d,clicks_90d,sessions_90d,content_age_days,ctr,avg_position,engagement_rate,trend_direction,is_declining_proxy
0,content_304f48230142,3803,29,17,187,0.76,10.6,5.88,down,1
1,content_a1fb4e703a9e,15320,7,9,445,0.05,20.3,0.00,down,1
2,content_9aa793d4d895,12581,11,11,141,0.09,36.5,0.00,down,1
3,content_331d6c4de07b,11751,58,78,463,0.49,6.2,1.28,stable,0
4,content_d99b7a2d90ca,19140,24,145,263,0.13,44.0,0.00,down,1
5,content_d4084a4bc775,3970,1,5,147,0.03,8.5,0.00,down,1
6,content_9a34b442b552,20,0,1,90,0.00,7.0,0.00,down,1
7,content_a63219c6e95a,1724,1,28,445,0.06,21.2,3.57,stable,0
8,content_5e6c160719bc,32574,29,68,90,0.09,46.0,5.88,down,1
9,content_c27558df2b0c,1240,2,3,257,0.16,4.9,0.00,down,1


## 5. Why ML beats a fixed rule here

A fixed rule could find simple cases, such as pages that are old and have many
impressions. However, one rule may miss important relationships between the
available signals.

For example, low CTR has a different meaning for a page in position 2 than
for a page in position 18. An old page may not need attention if its traffic
and engagement are still strong. A declining page with very few impressions
may also be less important than a smaller decline on a high-demand page.

The ranking may need to combine several measured signals, including
impressions, clicks, CTR, average position, content age, sessions, engagement,
and content type. These relationships may be too complicated for one fixed
`if` statement.

Machine learning may combine these signals and create a better page ranking.
However, ML does not automatically beat a rule. I will first build a simple,
transparent baseline and compare it with the model using Precision@50.

ML will only be useful if it produces a better and explainable review queue.
The output is decision-support, and a human reviewer will still make the final
content decision.

In [11]:
signals = [
    "impressions",
    "clicks",
    "CTR",
    "average position",
    "content age",
    "sessions",
    "engagement",
    "content type",
]

print("The ranking may need to combine these signals:")

for signal in signals:
    print("-", signal)

print("\nML must be compared with a transparent fixed-rule baseline.")

The ranking may need to combine these signals:
- impressions
- clicks
- CTR
- average position
- content age
- sessions
- engagement
- content type

ML must be compared with a transparent fixed-rule baseline.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.